<a href="https://colab.research.google.com/github/ob-s/FedShield/blob/main/Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Loads datasets from drive folder
file_path = '/content/drive/MyDrive/Malware_Project/MalMem2022.csv'
df = pd.read_csv(file_path)

print(f"Total dataset size: {len(df)} items")

Total dataset size: 58596 items


In [4]:
# The dataset has a 'Class' column (Benign vs Malware)
#'Category' column detailing specific malware types (Spyware, Ransomware, etc.)
# For deep learning classifier, only the binary 'Class' column is needed.
if 'Category' in df.columns:
    df = df.drop(columns=['Category'])

# Convert string labels to binary integers (0 for Benign, 1 for Malware)
df['Class'] = df['Class'].apply(lambda x: 0 if x == 'Benign' else 1)

# Separate features (X) and labels (y)
X = df.drop(columns=['Class'])
y = df['Class']

# Drop non-numeric columns (such as sample names, hashes, or text identifiers)
X = X.select_dtypes(include=[np.number])

# Execute the 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set (80%): {X_train.shape[0]} items")
print(f"Testing set (20%): {X_test.shape[0]} items")

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Training set (80%): 46876 items
Testing set (20%): 11720 items


In [5]:
# Build the baseline classifier using TensorFlow
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

# Train the model
print("Training the baseline model...")
history = model.fit(X_train_scaled, y_train,
                    epochs=15,
                    batch_size=64,
                    validation_split=0.2)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training the baseline model...
Epoch 1/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.9981 - loss: 0.0173 - precision: 1.0000 - recall: 0.9981 - val_accuracy: 1.0000 - val_loss: 8.1038e-05 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 2/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 1.0000 - loss: 1.3472e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 1.0000 - val_loss: 1.9651e-05 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 3/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 4.5045e-05 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 1.0000 - val_loss: 8.3345e-06 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 4/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 2.0695e-05 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 1.0000 - val_loss: 4.3743e-06 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 5/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 1.3963e-05 - precision

In [6]:
# Test the model on the unseen 20% test data
results = model.evaluate(X_test_scaled, y_test, verbose=0)
loss = results[0]
accuracy = results[1]

print(f"Baseline Model Accuracy: {accuracy * 100:.2f}%")

Baseline Model Accuracy: 100.00%
